Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: 엔드포인트 생성 없이 파운데이션 모델 자체에 요청/응답 빅쿼리 자동 로깅을 설정하고 호출한다.

## 제미나이 API 요청 및 응답 빅쿼리 실시간 자동 로깅 (Gemini API Request-Response Logging )

### 1. 의존성 패키지 설치

빅쿼리 데이터세트 제어 및 최신 구글 GenAI SDK를 활용하기 위해 필요한 클라우드 라이브러리들을 설치한다.

In [ ]:
!pip install --quiet google-genai google-cloud-aiplatform google-cloud-bigquery

### 2. 활성 GCP 프로젝트 ID 동적 탐색 및 설정

현재 활성화되어 있는 사용자의 자격 증명을 기반으로 프로젝트 ID를 탐색하고, 로그가 적재될 빅쿼리 데이터세트 등의 전역 변수를 선언한다.

In [ ]:
import google.auth

BIGQUERY_DATASET_ID = "gcp_logs"
LOCATION_ID = "global"
MODEL_ID = "gemini-3.5-flash"

try:
  _, project_id = google.auth.default()
  if not project_id:
    raise ValueError("프로젝트 ID를 탐색하지 못했다.")
  print(f"[성공] 활성화된 GCP 프로젝트 ID 감지: {project_id}")
except Exception as e:
  print(f"[경고] 자격 증명을 통해 프로젝트 ID를 찾지 못했다. 수동으로 설정해야 한다. (GCP 콘솔 자격증명 참조 주소: https://console.cloud.google.com/ )")
  project_id = "your-project-id" # 본인의 실제 GCP 프로젝트 ID로 변경하기 바란다.

### 3. 파운데이션 모델 로깅 활성화 설정 및 테스트 예측 호출

빅쿼리 목적지 데이터세트를 생성한 뒤, 파운데이션 모델 자체에 자동 로깅 기능을 바인딩하고 신형 GenAI SDK로 예측 호출을 구동한다.

**상세 파이프라인 제어 흐름:**
* **Vertex AI 환경 초기화**: 제공된 프로젝트 ID와 글로벌 리전을 기준으로 Vertex AI 환경을 활성화한다.
* **빅쿼리 목적지 데이터세트 생성**: 로그를 안전하게 수집할 데이터세트가 없는 경우, 미국(`US` ) 멀티 리전에 데이터세트를 자동 생성한다.
* **실시간 자동 로깅 활성화**: `GenerativeModel` 클래스의 `set_request_response_logging_config` 기능을 호출하여 샘플링 레이트 1.0(100% 로깅 )으로 빅쿼리 실시간 스트리밍 적재를 바인딩한다.
* **신규 GenAI SDK 예측 호출**: 신형 `google-genai` 라이브러리의 `generate_content`를 통해 가벼운 제미나이 3.5 플래시 모델을 호출하고 응답 결과를 확인한다.

In [ ]:
from google import genai
from google.cloud import bigquery
from google.genai import types
from vertexai.preview.generative_models import GenerativeModel
import vertexai

vertexai.init(project=project_id, location=LOCATION_ID)

try:
  bq_client = bigquery.Client(project=project_id)
  dataset_ref = bq_client.dataset(BIGQUERY_DATASET_ID)
  dataset = bigquery.Dataset(dataset_ref)
  dataset.location = "US"
  bq_client.create_dataset(dataset, exists_ok=True)
  print(f"[성공] 빅쿼리 데이터세트가 준비되었다: {BIGQUERY_DATASET_ID}")
except Exception as e:
  print(f"[안내] 데이터세트 사전 생성 중 예외가 발생했으나 계속 진행한다: {e}")

control_model = GenerativeModel(MODEL_ID)

try:
  control_model.set_request_response_logging_config(
    enabled=True,
    sampling_rate=1.0,
    bigquery_destination=f"bq://{project_id}.{BIGQUERY_DATASET_ID}"
  )
  print("[성공] 파운데이션 모델에 실시간 자동 로깅 설정을 바인딩했다.")
except Exception as e:
  print(f"[안내] 로깅 설정 활성화 중 예외가 발생했으나 계속 진행한다 (이미 설정되었을 수 있음): {e}")

client = genai.Client(
  vertexai=True,
  project=project_id,
  location=LOCATION_ID
)

response = client.models.generate_content(
  model=MODEL_ID,
  contents="GCP의 주요 특징을 두 가지만 요약해라."
)

print("\n=== [제미나이 답변 결과] ===")
print(response.text)
print("============================\n")

### 4. 빅쿼리에 자동 생성된 테이블 감지 및 사용자별 토큰 사용량 집계 조회

로깅 파이프라인에서 적재가 지연될 시간을 고려하여 잠시 대기한 후, 생성된 테이블을 탐색 및 분석 쿼리를 수행한다.

**빅쿼리 분석 흐름 및 최적화 규칙:**
* **로깅 파이프라인 적재 대기**: 스트리밍 로그가 빅쿼리 테이블로 안전하게 플러시되도록 15초간 대기한다.
* **자동 생성 테이블 탐색**: 데이터세트 내부의 테이블 목록을 순회하며 접두사 `logging_` 또는 접미사 `_logging`이 포함된 자동 생성 로그 테이블명을 탐색한다.
* **빅쿼리 쿼리 비용 최적화**: 대량의 로그 데이터를 스캔하여 불필요한 빅쿼리 과금이 발생하는 일을 원천 방지하기 위해, 파티션 컬럼인 `logging_time` 필터를 쿼리에 사용하여 최근 7일 동안 수집된 데이터만 제한 스캔하도록 강제 최적화(Partition Pruning )한다.

In [ ]:
import time
import sys

print("[안내] 로깅 파이프라인에서 빅쿼리 테이블로 데이터가 적재되는 시간을 확보하기 위해 15초 대기한다...")
time.sleep(15)

try:
  bq_client = bigquery.Client(project=project_id)
  dataset_ref = bq_client.dataset(BIGQUERY_DATASET_ID)
  tables = bq_client.list_tables(dataset_ref)
  
  table_name = None
  for table in tables:
    if "logging_" in table.table_id or "_logging" in table.table_id: 
      table_name = table.table_id
      break
      
  if table_name:
    print(f"[성공] 자동 생성된 로그 테이블 감지: {table_name}")
    print("[자가 분석 시작] 해당 테이블에서 사용자별 토큰 점유율을 조회한다...")
    
    query = f"""
      SELECT COUNT(1) as call_count,
             SUM(LAX_INT64(full_response.usageMetadata.candidatesTokenCount)) as total_candidate_tokens,
             SUM(LAX_INT64(full_response.usageMetadata.promptTokenCount)) as total_prompt_tokens,
             SUM(LAX_INT64(full_response.usageMetadata.totalTokenCount)) as total_tokens,
             COALESCE(JSON_EXTRACT_SCALAR(full_request, '$.labels.principal_email'), 'default_user') as user_email
        FROM `{project_id}.{BIGQUERY_DATASET_ID}.{table_name}`
       WHERE logging_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
       GROUP BY user_email
       ORDER BY total_tokens DESC;
    """
    
    query_job = bq_client.query(query)
    results = query_job.result()
    
    print("\n=== [사용자별 토큰 통계 분석 결과] ===")
    for row in results:
      print(f"사용자 계정: {row.user_email}")
      print(f"  - 호출 횟수: {row.call_count}회")
      print(f"  - 프롬프트 토큰 합계: {row.total_prompt_tokens}")
      print(f"  - 답변 생성 토큰 합계: {row.total_candidate_tokens}")
      print(f"  - 전체 사용 토큰 합계: {row.total_tokens}")
      print("-"*48)
  else:
    print("[경고] 자동 생성된 로그 테이블을 감지하지 못했다. 적재 지연일 수 있으니 잠시 후 다시 조회를 권장한다.")
except Exception as e:
  print(f"[오류] 빅쿼리 로그를 조회하는 도중 에러가 발생했다: {e}")